# Create three Pandas DataFrames

In [3]:
import pandas as pd

- This is the basic *carsales* DataFrame extended with two more Sales places plus the index column transformed to a feature column.

In [10]:
cardata = { "Mercedes": [2, 4, 0, 4, 0, 3],
           "Ford": [3, 0, 0, 1, 6, 12],
           "Tata":[9, 3, 4, 1, 0, 0],
           "Renault":[12, 1, 0, 0, 3, 1]}

carsales = pd.DataFrame(cardata)
carsales.rename(index={0: "One",
                       1: "Two",
                       2: "Three",
                       3: "Four",
                       4: "Five",
                       5: "Six"},
                inplace=True)
carsales.insert(0,
                "Sales_place_name",
                ["Europe 1", "Australia 1", "USA 1", "Asia 1", "Africa 1", "South America 1"],
                allow_duplicates=True)

carsales2 = pd.DataFrame({"Sales_place_name": ["South America 1", "Asia 1"],
                          "Mercedes": [3, 4],
                          "Ford": [2, 1],
                          "Tata": [1, 1],
                          "Renault": [1, 0]})
carsales2.rename(index={0: "Seven",
                        1: "Eight"},
                 inplace=True)

carsales = pd.concat([carsales, carsales2])
carsales.index.rename("Sales place", inplace=True)
carsales.reset_index(inplace=True)  # Turns index column into a feature column
carsales

,Sales place,Sales_place_name,Mercedes,Ford,Tata,Renault
0,One,Europe 1,2,3,9,12
1,Two,Australia 1,4,0,3,1
2,Three,USA 1,0,0,4,0
3,Four,Asia 1,4,1,1,0
4,Five,Africa 1,0,6,0,3
5,Six,South America 1,3,12,0,1
6,Seven,South America 1,3,2,1,1
7,Eight,Asia 1,4,1,1,0


- This DataFrame includes Car inventory data. Unique column values for "Sales_place_name", all included in Carsales and fewer labels than in the Carsales DataFrame.

In [6]:
invData = pd.DataFrame({"Sales_place_name": ["Europe 1", "Australia 1", "USA 1", "Asia 1", "Africa 1", "South America 1"],
                        "Car_inv": [132, 54, 323, 267, 183, 172]})
invData

,Sales_place_name,Car_inv
0,Europe 1,132
1,Australia 1,54
2,USA 1,323
3,Asia 1,267
4,Africa 1,183
5,South America 1,172


- This DataFrame includes Car inventory data. Non-unique Column values for "Sales_place_name", some included in Carsales and more labels than in the Carsales DataFrame.

In [11]:
invData2 = pd.DataFrame({"Sales_place_name": ["Europe 1", "Europe 1", "Australia 1", "USA 1", "Asia 1", "Canada 1", "Africa 1", "South America 1", "South America 2"],
                          "Car_inv": [132, 131, 54, 323, 267, 45, 183, 172, 144]})
invData2

,Sales_place_name,Car_inv
0,Europe 1,132
1,Europe 1,131
2,Australia 1,54
3,USA 1,323
4,Asia 1,267
5,Canada 1,45
6,Africa 1,183
7,South America 1,172
8,South America 2,144


# Standard inner join merge

In [13]:
carsales3 = carsales.merge(invData, how="inner", on=None, left_on="Sales_place_name", right_on="Sales_place_name")
carsales3

,Sales place,Sales_place_name,Mercedes,Ford,Tata,Renault,Car_inv
0,One,Europe 1,2,3,9,12,132
1,Two,Australia 1,4,0,3,1,54
2,Three,USA 1,0,0,4,0,323
3,Four,Asia 1,4,1,1,0,267
4,Eight,Asia 1,4,1,1,0,267
5,Five,Africa 1,0,6,0,3,183
6,Six,South America 1,3,12,0,1,172
7,Seven,South America 1,3,2,1,1,172


*Sales place* is still unique indentifier for the rows. The DataFrame has been sorted "behind the scenes". *Asia1* has been moved from the end in the original *carsales* DF.<br>
The *Car_inv* column contains correct data but the data is considered to be implicit duplicates, as there are two excessive *Sales_place_name* duplicates.


## Understanding Pandas Inner Joins with Duplicate Keys

In [14]:
carsales4 = carsales.merge(invData2, how="inner", on=None, left_on="Sales_place_name", right_on="Sales_place_name")
carsales4

,Sales place,Sales_place_name,Mercedes,Ford,Tata,Renault,Car_inv
0,One,Europe 1,2,3,9,12,132
1,One,Europe 1,2,3,9,12,131
2,Two,Australia 1,4,0,3,1,54
3,Three,USA 1,0,0,4,0,323
4,Four,Asia 1,4,1,1,0,267
5,Eight,Asia 1,4,1,1,0,267
6,Five,Africa 1,0,6,0,3,183
7,Six,South America 1,3,12,0,1,172
8,Seven,South America 1,3,2,1,1,172


When performing an **inner merge** in pandas on a key that has duplicate values in one or both DataFrames, every matching row in the left DataFrame is paired with every matching row in the right DataFrame. This can lead to more rows than you might initially expect.

**Example Scenario**  
- **Left table (`carsales`)** has:  
  - `Europe 1`: 1 row  
  - `Australia 1`, `USA 1`, `Africa 1`: 1 row each  
  - `Asia 1`, `South America 1`: 2 rows each  

- **Right table (`invData2`)** has:  
  - `Europe 1`: 2 rows  
  - `Australia 1`, `USA 1`, `Asia 1`, `Africa 1`, `South America 1`, `South America 2`, `Canada 1`: varied counts  

**How the Math Works**

| Key               | Left count × Right count | Resulting rows |
|-------------------|---------------------------|----------------|
| `Europe 1`        | 1 × 2                     | 2              |
| `Australia 1`     | 1 × 1                     | 1              |
| `USA 1`           | 1 × 1                     | 1              |
| `Africa 1`        | 1 × 1                     | 1              |
| `Asia 1`          | 2 × 1                     | 2              |
| `South America 1` | 2 × 1                     | 2              |

Total from shared keys = 2 + 1 + 1 + 1 + 2 + 2 = **9 rows**

> **Note:**  
> Keys present in only one table (e.g., `Canada 1`, `South America 2`) are **excluded** completely in an inner join.

---

# Standard left join merge

In [16]:
carsales5 = carsales.merge(invData, how="left", on=None, left_on="Sales_place_name", right_on="Sales_place_name")
carsales5

,Sales place,Sales_place_name,Mercedes,Ford,Tata,Renault,Car_inv
0,One,Europe 1,2,3,9,12,132
1,Two,Australia 1,4,0,3,1,54
2,Three,USA 1,0,0,4,0,323
3,Four,Asia 1,4,1,1,0,267
4,Five,Africa 1,0,6,0,3,183
5,Six,South America 1,3,12,0,1,172
6,Seven,South America 1,3,2,1,1,172
7,Eight,Asia 1,4,1,1,0,267


The **left** join preserved the order of the rows. The implicit duplicates in the *Car_inv* column remain.<br>
Normally, the **inner join** and the **left join** would be considered the correct join methods for such FataFrames.

In [17]:
carsales6 = carsales.merge(invData2, how="left", on=None, left_on= "Sales_place_name", right_on="Sales_place_name")
carsales6

,Sales place,Sales_place_name,Mercedes,Ford,Tata,Renault,Car_inv
0,One,Europe 1,2,3,9,12,132
1,One,Europe 1,2,3,9,12,131
2,Two,Australia 1,4,0,3,1,54
3,Three,USA 1,0,0,4,0,323
4,Four,Asia 1,4,1,1,0,267
5,Five,Africa 1,0,6,0,3,183
6,Six,South America 1,3,12,0,1,172
7,Seven,South America 1,3,2,1,1,172
8,Eight,Asia 1,4,1,1,0,267


# Standard right join merge

In [19]:
carsales6 = carsales.merge(invData, how="right", on=None, left_on="Sales_place_name", right_on="Sales_place_name")
carsales6

,Sales place,Sales_place_name,Mercedes,Ford,Tata,Renault,Car_inv
0,One,Europe 1,2,3,9,12,132
1,Two,Australia 1,4,0,3,1,54
2,Three,USA 1,0,0,4,0,323
3,Four,Asia 1,4,1,1,0,267
4,Eight,Asia 1,4,1,1,0,267
5,Five,Africa 1,0,6,0,3,183
6,Six,South America 1,3,12,0,1,172
7,Seven,South America 1,3,2,1,1,172


The merged DataFrames are sorted on the *invData* DataFrame.

In [22]:
carsales7 = carsales.merge(invData2, how="right", on=None, left_on="Sales_place_name", right_on="Sales_place_name")
carsales7

,Sales place,Sales_place_name,Mercedes,Ford,Tata,Renault,Car_inv
0,One,Europe 1,2.0,3.0,9.0,12.0,132
1,One,Europe 1,2.0,3.0,9.0,12.0,131
2,Two,Australia 1,4.0,0.0,3.0,1.0,54
3,Three,USA 1,0.0,0.0,4.0,0.0,323
4,Four,Asia 1,4.0,1.0,1.0,0.0,267
5,Eight,Asia 1,4.0,1.0,1.0,0.0,267
6,NaN,Canada 1,NaN,NaN,NaN,NaN,45
7,Five,Africa 1,0.0,6.0,0.0,3.0,183
8,Six,South America 1,3.0,12.0,0.0,1.0,172
9,Seven,South America 1,3.0,2.0,1.0,1.0,172


Duplicates on *Sales place name* +3 rows. 2 rows mainly consist of missing values.

# Standard outer join

In [24]:
carsales8 = carsales.merge(invData, how="outer", on=None, left_on="Sales_place_name", right_on="Sales_place_name")
carsales8

,Sales place,Sales_place_name,Mercedes,Ford,Tata,Renault,Car_inv
0,One,Europe 1,2,3,9,12,132
1,Two,Australia 1,4,0,3,1,54
2,Three,USA 1,0,0,4,0,323
3,Four,Asia 1,4,1,1,0,267
4,Eight,Asia 1,4,1,1,0,267
5,Five,Africa 1,0,6,0,3,183
6,Six,South America 1,3,12,0,1,172
7,Seven,South America 1,3,2,1,1,172


In this case, the **outer join** is nearly equal to the **inner join** because of the data in the joining tables.

In [26]:
carsales9 = carsales.merge(invData2, how="outer", on=None, left_on="Sales_place_name", right_on="Sales_place_name")
carsales9

,Sales place,Sales_place_name,Mercedes,Ford,Tata,Renault,Car_inv
0,One,Europe 1,2.0,3.0,9.0,12.0,132
1,One,Europe 1,2.0,3.0,9.0,12.0,131
2,Two,Australia 1,4.0,0.0,3.0,1.0,54
3,Three,USA 1,0.0,0.0,4.0,0.0,323
4,Four,Asia 1,4.0,1.0,1.0,0.0,267
5,Eight,Asia 1,4.0,1.0,1.0,0.0,267
6,Five,Africa 1,0.0,6.0,0.0,3.0,183
7,Six,South America 1,3.0,12.0,0.0,1.0,172
8,Seven,South America 1,3.0,2.0,1.0,1.0,172
9,NaN,Canada 1,NaN,NaN,NaN,NaN,45


Duplicates on *Sales place name* +3 rows. 2 rows largely consisting of missing values.